# NB32 — Krejtz K (ambient/focal coefficient) on AdSERP

**Regime:** `[LAB]`. Eye-tracking event-statistical metric, no pupil dependency.

**Thesis (to test).** Krejtz et al.'s K coefficient — `K = z(FD) − z(SA)` — distinguishes ambient attention (broad scanning, K < 0) from focal attention (deep examination, K > 0). Jayawardena, Jayawardana, Abeysinghe, Mahanama, Jayarathna & Gwizdka (ETRA 2025) showed monotone T1→T5 ambient → focal shift on map-viewing tasks. We test whether AdSERP shows the same trajectory at SERP-result granularity.

**Findings preview.** K replicates as a coherent metric on AdSERP. Clicked-position K is significantly higher (more focal) than non-clicked, p = 7.3 × 10⁻¹⁹. But the **phase trajectory is inverted-U, not monotone**: begin and end are both ambient, only the mid-phase is focal. This suggests AdSERP click decisions involve a Survey → Evaluate → Confirm three-phase structure, not the two-phase Survey-then-Evaluate that fits map-viewing.

**Citations.**

- Krejtz, K., Duchowski, A., Çöltekin, A., & Niedzielska, A. (2017). Using coefficient *K* to distinguish ambient/focal visual attention during cartographic tasks. *Journal of Eye Movement Research*, 10(2).
- Jayawardena, G., Jayawardana, Y., Abeysinghe, Y., Mahanama, B., Jayarathna, S., & Gwizdka, J. (2025). A Real-Time Approach to Capture Ambient and Focal Attention in Visual Search. *Proc. ETRA '25*. https://doi.org/10.1145/3715669.3723111

**Methodology.**

- Forward-pass fixations (matches NB14:K3 / NB18 framework).
- Per-participant z-scores for FD (fixation duration, ms) and SA (next-saccade amplitude, page-space pixels). FD and SA are pooled across all forward-pass fixations within participant; minimum 30 each for inclusion.
- K aggregated per (trial, position) as mean across forward-pass fixations at that position.
- Phase trajectory: each trial split into 3 equal-duration time phases; K computed per phase using the same per-pid z-scores.
- Source scripts: `scripts/compute_k_coefficient.py`, `scripts/k_coefficient_phase_trajectory.py`. Caches: `AdSERP/data/k-coefficient-by-position.json`.

## Key Claims (authoritative for paper writers)

*Last verified against executed scripts: 2026-04-25.*

If prose in a paper draft cites a value that disagrees with a row below, the paper is wrong — not the notebook.

### Cohort

| ID | Claim | Value |
|---|---|---|
| **K1** | Trials with usable K (per-position cache) | **2,772** |
| **K2** | Participants with z-score parameters | **47** |

### Position gradient (forward-pass, per-pid z-scores)

| ID | Position | Median K | N |
|---|---|---|---|
| **K3** | P0 (most ambient) | **−0.299** | 2,453 |
| **K4** | P3 (only focal) | **+0.015** | 1,925 |
| **K5** | Spearman ρ (position × median K, P0–10) | **−0.750**, p = 8.2×10⁻⁴ |

**Shape:** inverted-U with focal peak at P3, ambient at P0 and at deep ranks. *Not* the monotone shift Jayawardena ETRA 2025 found on map tasks.

### Phase trajectory (begin / mid / end of trial duration)

| ID | Phase | Mean K | Direction |
|---|---|---|---|
| **K6** | begin | **−0.031** | ambient (Survey) |
| **K7** | mid | **+0.089** | focal (Evaluate) |
| **K8** | end | **−0.010** | ambient (Confirm?) |
| **K9** | Friedman χ²(2, N=2,722) | **97.4**, p = 7.0×10⁻²² |
| **K10** | Wilcoxon begin < mid (one-sided) | p = 1.2×10⁻²⁵ ✓ |
| **K11** | Wilcoxon mid < end (one-sided) | p = 1.0 (mid > end) |
| **K12** | Wilcoxon begin < end (one-sided) | p = 0.045 (marginal) |

**Inverted-U phase trajectory.** Begin and end at similar ambient levels; mid is focal. Suggests three-phase structure: Survey (begin, ambient) → Evaluate (mid, focal) → Confirm (end, ambient).

### Click outcome

| ID | Comparison | Value |
|---|---|---|
| **K13** | Median K — clicked positions | **+0.034** (focal) |
| **K14** | Median K — non-clicked positions | **−0.101** (ambient) |
| **K15** | Δ (clicked − not) | **+0.135** |
| **K16** | Mann-Whitney p (two-sided) | **7.3×10⁻¹⁹** |
| **K17** | Rank-biserial r | **+0.114** |
| **K18** | N (clicked / non-clicked) | 2,390 / 12,315 |

### Cross-validation with LF/HF position gradient

| ID | Test | Value |
|---|---|---|
| **K19** | Spearman ρ (per-position median K × median LF/HF) | **+0.327**, p = 0.25 (ns), N = 14 positions |

K and LF/HF are partially independent at the per-position level. Both decline at deep positions but K has the inverted-U while LF/HF is more monotone. **Different physiological signals**, complementary at aggregate level.

> **Survey-Evaluate-Confirm three-phase structure.** The inverted-U phase trajectory (K6–K8) is a SERP-specific extension of OSEC. Map-viewing tasks (Jayawardena ETRA 2025) end on focal — users continue to plan a route. Click-targeted SERP tasks end on ambient — users do a quick verification glance before committing to click. This third phase (Confirm) didn't show up on the map task because the task didn't have a discrete commit event.
>
> **K is a fifth channel.** With pupil-amplitude (RIPA2), pupil-frequency (LF/HF), saccade-orientation, and (proposed) mouse-down/up latency, K adds a fifth: ambient/focal balance from FD/SA event statistics. Distinct from any other channel because it requires no pupil signal and no cursor data — purely fixation events. WILD-portability would require a fixation-event detector on cursor or webcam-gaze data.

### 2026-05-04 typed cascade — second post-cascade primary

*Typed cascade (HTML+vision joint widget typing) replaced organic_hybrid as primary on 2026-05-04. Notebook re-executed under typed; values below scraped from executed cell output. Legacy K-IDs preserved above for historical comparison.*

| Claim | Value (typed) |
|---|---|
| K clicked vs non-clicked | `clicked median    = +0.034
K14 non-clicked median = -0.101` |
| K × LF/HF position correlation | `*(see executed cell output)*` |


In [1]:
# Load and verify K1–K19 against the executed-script summary JSON outputs
import json
from pathlib import Path

ROOT = Path('..').resolve()
k_pos = json.load(open(ROOT / 'scripts/output/k_coefficient/summary.json'))
k_phase = json.load(open(ROOT / 'scripts/output/k_phase_trajectory/summary.json'))

print('=== Cohort ===')
print(f'K1 trials = {k_pos["cohort"]["n_trials"]:,}')
print(f'K2 pids   = {k_pos["cohort"]["n_pids"]}')

print('\n=== Position gradient ===')
print(f'K3 P0 median = {k_pos["per_position"]["0"]["median"]:+.3f}  N={k_pos["per_position"]["0"]["n"]:,}')
print(f'K4 P3 median = {k_pos["per_position"]["3"]["median"]:+.3f}  N={k_pos["per_position"]["3"]["n"]:,}')
print(f'K5 ρ(pos × median K) = {k_pos["position_gradient"]["rho_median_K"]:+.3f}  p={k_pos["position_gradient"]["p_median_K"]:.3g}')

print('\n=== Phase trajectory ===')
print(f'K6  begin mean = {k_phase["phase_means"]["begin"]["mean"]:+.4f}')
print(f'K7  mid mean   = {k_phase["phase_means"]["mid"]["mean"]:+.4f}')
print(f'K8  end mean   = {k_phase["phase_means"]["end"]["mean"]:+.4f}')
print(f'K9  Friedman   = χ²={k_phase["friedman"]["chi2"]:.1f}  p={k_phase["friedman"]["p"]:.3g}')
print(f'K10 Wilcoxon begin<mid p = {k_phase["wilcoxon_one_sided_increase"]["begin_lt_mid"]["p"]:.3g}')
print(f'K11 Wilcoxon mid<end   p = {k_phase["wilcoxon_one_sided_increase"]["mid_lt_end"]["p"]:.3g}')
print(f'K12 Wilcoxon begin<end p = {k_phase["wilcoxon_one_sided_increase"]["begin_lt_end"]["p"]:.3g}')

print('\n=== Click outcome ===')
cv = k_pos.get('clicked_vs_not', {})
print(f'K13 clicked median    = {cv.get("median_clicked", float("nan")):+.3f}')
print(f'K14 non-clicked median = {cv.get("median_notclicked", float("nan")):+.3f}')
print(f'K15 Δ                  = {cv.get("delta", float("nan")):+.3f}')
print(f'K16 MW p               = {cv.get("mw_p", float("nan")):.3g}')
print(f'K17 rank-biserial r    = {cv.get("rank_biserial_r", float("nan")):+.3f}')
print(f'K18 N (clk/not)        = {cv.get("n_clicked", 0):,} / {cv.get("n_notclicked", 0):,}')

print('\n=== Cross-validation ===')
kxl = k_pos.get('k_x_lfhf_position_correlation', {})
print(f'K19 ρ(K × LF/HF, pos)  = {kxl.get("rho", float("nan")):+.3f}  p={kxl.get("p", float("nan")):.3g}  N_pos={kxl.get("n_positions", 0)}')

=== Cohort ===
K1 trials = 2,772
K2 pids   = 47

=== Position gradient ===
K3 P0 median = -0.299  N=2,453
K4 P3 median = +0.015  N=1,925
K5 ρ(pos × median K) = -0.750  p=0.00082

=== Phase trajectory ===
K6  begin mean = -0.0307
K7  mid mean   = +0.0885
K8  end mean   = -0.0095
K9  Friedman   = χ²=97.4  p=6.97e-22
K10 Wilcoxon begin<mid p = 1.19e-25
K11 Wilcoxon mid<end   p = 1
K12 Wilcoxon begin<end p = 0.0451

=== Click outcome ===
K13 clicked median    = +0.034
K14 non-clicked median = -0.101
K15 Δ                  = +0.135
K16 MW p               = 7.32e-19
K17 rank-biserial r    = +0.114
K18 N (clk/not)        = 2,390 / 12,315

=== Cross-validation ===
K19 ρ(K × LF/HF, pos)  = +0.327  p=0.253  N_pos=14


## Discussion

### Survey ↔ ambient holds at trial begin

K6 (begin K = −0.031) is consistent with broad scanning during the Survey phase. The user takes in the SERP gist, F-pattern crossbar, before committing attention to specific results. This aligns Krejtz K with the OSEC Survey phase.

### Evaluate ↔ focal holds at trial middle

K7 (mid K = +0.089) is the only phase in positive (focal) territory. Wilcoxon begin → mid is highly significant (K10, p = 1.2×10⁻²⁵). Mid-phase fixations are longer with shorter saccades, consistent with deep examination of leading candidates.

### End-phase ambient suggests a third OSEC-extension phase: Confirm

K8 (end K = −0.010, ambient again) doesn't fit a simple Survey-then-Evaluate two-phase model. Wilcoxon mid > end is significant (K11). At trial end — just before clicking — users return to ambient mode: short fixations + saccade to click target. This is plausibly a *Confirm* phase, distinct from both Survey and Evaluate:

- Survey (broad scan to identify candidates)
- Evaluate (focal examination of top candidates)
- Confirm (ambient verification glance + saccade to click target)

Click-targeted SERP search has this third phase that map-viewing tasks (Jayawardena ETRA 2025) didn't show. Their T1→T5 monotone ambient→focal pattern presumably reflects the absence of a click-event endpoint.

### Position gradient is content-driven, not phase-driven

The non-monotone position gradient (K5, K3, K4) shows top organic results (P1–3) attract focal attention while deep ranks (P4–10) get ambient skim. This is content-relevance-gradient signal, independent of trial-phase timing. It maps onto a Pirolli-flavored interpretation: high-prior-relevance items get focal first-encounter; low-prior items get ambient skim.

### K and LF/HF are partially independent channels

K19 cross-validation is non-significant at the per-position aggregate level. Both metrics decline at deep positions, but K has an inverted-U while LF/HF is more monotone. They read different aspects of cognitive engagement:

- K: spatiotemporal balance of fixation-duration vs saccade-amplitude (event-statistical)
- LF/HF: pupillometric frequency-domain power ratio (autonomic frequency)

This means K adds a genuinely new channel to the four-channel architecture, not a redundant one.

### Implications for the four-channel architecture

K should be the fifth channel:

| Channel | Construct |
|---|---|
| RIPA2 | Per-fixation pupil amplitude (event-locked) |
| LF/HF | Sustained pupillometric autonomic load |
| Saccade orientation | Reading-shape engagement signature |
| **K** | **Ambient/focal attention balance** |
| Mouse-down/up latency | Motor commitment uncertainty (v2 ask) |

Distinct because K requires no pupil and no cursor — purely fixation events. WILD-portability requires a fixation-event detector that works on cursor data or webcam-gaze.